# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and their fields, using `@id` references only.

In [ ]:
# List all record sets and their fields using @id
print("Available Record Sets and their Fields (showing @id):\n")
record_set_ids = []
for rset in dataset.record_sets:
    print(f"Record Set: {rset['@id']} | name: {rset.get('name','')}")
    record_set_ids.append(rset['@id'])
    if 'field' in rset and rset['field']:
        for field in rset['field']:
            # field can be {'@id': ..., ...} or a field @id
            fid = field if isinstance(field, str) else field.get('@id', field)
            print(f"  Field: {fid}")
    print('-'*60)

# Display the found record sets
print(f"All Record Set @id values: {record_set_ids}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# -- Choose record set for analysis --
# For this notebook, select the main protein table (most likely primary table in proteomics dataset)
# If there are multiple recordSets, choose the main one by name or @id. Adjust @id as needed after overview.

# We'll take the first (or only) record set for demonstration
main_record_set_id = record_set_ids[0]

# Extract all records for this record set
records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(records)

print(f'Loaded {len(df)} records from record set: {main_record_set_id}')
print('Columns (@id):')
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

The following examples select a numeric field (e.g., coverage or MW), filter records, normalize values, and optionally group or aggregate. Adjust field `@id` variables as needed based on your record set.

In [ ]:
# Pick a prominent numeric field - for FAIR^2 proteomics, likely one of:
#  - cr:coverage    (peptide/protein coverage percentage)
#  - cr:MW_da       (molecular weight)
#  - cr:peptides    (number of peptides)
#
# Adjust the field @id if needed after examining columns above (~'cr:...' syntax)

# Set key field @id names for EDA (edit as needed with your actual column @id)
numeric_field = 'cr:coverage'  # coverage percentage, for example
group_field = 'cr:sample'      # group by sample, if present, else e.g., 'cr:accession'
if numeric_field not in df.columns:
    # Try alternative field name ({cr:MW_da} or {cr:peptides})
    if 'cr:MW_da' in df.columns:
        numeric_field = 'cr:MW_da'
    elif 'cr:peptides' in df.columns:
        numeric_field = 'cr:peptides'

# Ensure the field is numeric
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

# Example filter and normalization: coverage > 10% or MW > 10,000 Da
if numeric_field == 'cr:coverage':
    threshold = 10
elif numeric_field == 'cr:MW_da':
    threshold = 10000
elif numeric_field == 'cr:peptides':
    threshold = 2
else:
    threshold = df[numeric_field].quantile(0.10)  # use 10th percentile as example

filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / (filtered_df[numeric_field].std() + 1e-8)
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Grouped summary (group by sample column if present, else by another categorical)
if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nGrouped data by {group_field} (mean {numeric_field}):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Distribution plot of the selected numeric field after filtering
plt.figure(figsize=(8, 4))
plt.hist(filtered_df[numeric_field].dropna(), bins=30, color='deepskyblue', edgecolor='k', alpha=0.7)
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.title(f'Distribution of {numeric_field} (filtered, > {threshold})')
plt.show()

# If grouped summaries are available, show bar chart
if 'grouped_df' in locals():
    plt.figure(figsize=(10, 4))
    plt.bar(grouped_df[group_field].astype(str), grouped_df[numeric_field], color='salmon')
    plt.xlabel(group_field)
    plt.ylabel(f'Mean {numeric_field}')
    plt.title(f'Mean {numeric_field} by {group_field} (filtered)')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

# Scatter Example (if comparing two numeric columns)
other_numeric = None
for col in ['cr:MW_da', 'cr:peptides', 'cr:coverage']:
    if col != numeric_field and col in filtered_df.columns:
        other_numeric = col
        break

if other_numeric:
    plt.figure(figsize=(6, 5))
    plt.scatter(filtered_df[numeric_field], filtered_df[other_numeric], alpha=0.5)
    plt.xlabel(numeric_field)
    plt.ylabel(other_numeric)
    plt.title(f'{numeric_field} vs {other_numeric} (filtered)')
    plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR² mass spectrometry dataset using the `mlcroissant` library.

- Loaded dataset schema and records dynamically from the Croissant `@id` references.
- Demonstrated overview of record sets and their field `@id`s for transparent, schema-driven data processing.
- Extracted a main record set to DataFrame and performed filtering, normalization, grouping, and several visualizations.

**Next steps:**
- Refine field selections to your analytical target (e.g., specific proteins or samples).
- Extend EDA and statistics based on your use case, considering interactive visualization, outlier detection, or advanced grouping.
- Reference the Croissant documentation for advanced API usage: https://mlcommons.github.io/croissant/python.html
